# Notebook 5 — Least squares: Gauss–Newton, and the damping that rescues it

**Day 5.** Read this after Lecture 5, with `ExpDecay`, `GaussNewton` and
`LevenbergMarquardt` written.

Curve fitting is the one place in this course where the objective has *structure* you
can exploit. Minimising

$$f(x) = \tfrac{1}{2}\|r(x)\|^2, \qquad r : \mathbb{R}^d \to \mathbb{R}^m$$

is not a general smooth minimisation, because the gradient and Hessian are

$$\nabla f = J^\top r, \qquad
\nabla^2 f = \underbrace{J^\top J}_{\text{first derivatives only}} \;+\;
\underbrace{\sum_i r_i \nabla^2 r_i}_{\text{needs second derivatives}} .$$

Gauss–Newton throws the second term away and solves $(J^\top J)\,\delta = -J^\top r$.
That buys a near-Newton method from first derivatives alone. This notebook measures the
bill:

1. **when the dropped term is negligible** — and it is not a matter of opinion, we
   measure its size;
2. **Gauss–Newton succeeding**, then failing in two entirely different ways;
3. **Levenberg–Marquardt** surviving every start that killed it, and *why*: we watch
   $\rho$ reject the exact step that produced the `nan`;
4. **what $\lambda$ and $\rho$ do**, iteration by iteration;
5. **breaking LM too** — because $\lambda \to \infty$ is an absorbing state.

> **A note on the interface.** `ILeastSquaresProblem` is deliberately *not* an
> `IObjective`. It offers `residuals` and `jacobian`, not `value` and `gradient`. A class
> that only knows how to minimise scalars cannot consume it, and that is the point: the
> segregation is what lets `GaussNewton` demand the structure it needs at the type level
> instead of hoping for it at run time.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# The optlab root is the nearest ancestor holding pyproject.toml, so this works whether
# Jupyter was started in notebooks/ or in the repository root.
HERE = Path.cwd()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "pyproject.toml").exists()), HERE.parent)

try:
    import optlab
except ModuleNotFoundError:
    sys.path.insert(0, str(ROOT / "src"))
    import optlab

sys.path.insert(0, str(ROOT))  # for datasets/

np.set_printoptions(precision=6, suppress=True)
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
                     "axes.titlesize": 10, "figure.dpi": 110})
rng = np.random.default_rng(20250921)

print("optlab root  :", ROOT)
print("optlab loaded:", Path(optlab.__file__).parent)

In [ ]:
def status(name, thunk):
    """Report whether one piece of the package is implemented, without a traceback."""
    try:
        thunk()
    except NotImplementedError:
        return f"  MISSING   {name}"
    except Exception as err:                      # noqa: BLE001 - we want to see anything
        return f"  BROKEN    {name}   ({type(err).__name__}: {err})"
    return f"  ok        {name}"

In [ ]:
from optlab.linalg import CholeskySolver
from optlab.observers import History
from optlab.optimizers import GaussNewton, LevenbergMarquardt
from optlab.problems.curve_fitting import ExpDecay, GaussianPeak, Sinusoid

_t = np.linspace(0.0, 1.0, 5)
_p = ExpDecay(_t, np.zeros(5))
_x = np.array([1.0, 1.0, 0.0])

print("Day 5 readiness")
print(status("ExpDecay.residuals", lambda: _p.residuals(_x)))
print(status("ExpDecay.jacobian", lambda: _p.jacobian(_x)))
print(status("GaussNewton",
             lambda: GaussNewton(CholeskySolver(), max_iter=1).minimize(_p, _x)))
print(status("LevenbergMarquardt",
             lambda: LevenbergMarquardt(CholeskySolver(), max_iter=1).minimize(_p, _x)))
print()
print("Used later in this notebook")
print(status("GaussianPeak.jacobian",
             lambda: GaussianPeak(_t, np.zeros(5)).jacobian(np.array([1.0, 0.5, 0.2]))))
print(status("Sinusoid.jacobian",
             lambda: Sinusoid(_t, np.zeros(5)).jacobian(np.array([1.0, 1.0, 0.0]))))
print()
print("From day 4")
print(status("CholeskySolver", lambda: CholeskySolver().solve(np.eye(2), np.ones(2))))

## 1. How big is the term Gauss–Newton throws away?

The justification you will read everywhere is "valid for small residuals". That is a
statement about a *number*, so let us produce the number.

We generate $y = a e^{-bt} + c$ with $(a, b, c) = (5, 1.3, 1)$ and add noise of a chosen
size. At the fitted solution we compare the two halves of the Hessian in spectral norm:

$$\text{ratio} \;=\; \frac{\bigl\|\sum_i r_i \nabla^2 r_i\bigr\|}{\|J^\top J\|} .$$

We do not need a hand-derived $\nabla^2 r_i$ for this. Differentiating the *Jacobian*
numerically gives the same thing and uses only the method you already wrote — which also
means this cell is a second, independent check that your `jacobian` is correct.

In [ ]:
x_true = np.array([5.0, 1.3, 1.0])              # a, b, c
t_grid = np.linspace(0.0, 4.0, 60)


def make_problem(noise, seed=3):
    gen = np.random.default_rng(seed)
    y = x_true[0] * np.exp(-x_true[1] * t_grid) + x_true[2]
    y = y + gen.normal(scale=noise, size=t_grid.size)
    return ExpDecay(t_grid, y), y


def dropped_term(problem, x, h=1e-5):
    """S = sum_i r_i * grad^2 r_i, by central differences on YOUR jacobian."""
    r = problem.residuals(x)
    d = x.size
    S = np.zeros((d, d))
    for k in range(d):
        e = np.zeros(d)
        e[k] = h
        dJ = (problem.jacobian(x + e) - problem.jacobian(x - e)) / (2.0 * h)
        S[:, k] = dJ.T @ r
    return 0.5 * (S + S.T)                       # symmetrise away the FD asymmetry


noises = [0.0, 0.02, 0.05, 0.2, 0.5, 1.0, 2.0]
ratios, gn_iters, lm_iters, costs = [], [], [], []
easy_start = np.array([4.0, 1.0, 0.5])

print(f"{'noise':>7} {'f(x*)':>12} {'||S||/||JtJ||':>15} {'GN iters':>9} {'LM iters':>9}")
for noise in noises:
    prob, _ = make_problem(noise)
    r_gn = GaussNewton(CholeskySolver(), tol=1e-6, max_iter=300).minimize(prob, easy_start)
    r_lm = LevenbergMarquardt(CholeskySolver(), tol=1e-6, max_iter=300).minimize(prob, easy_start)
    J = prob.jacobian(r_lm.x)
    ratio = np.linalg.norm(dropped_term(prob, r_lm.x), 2) / np.linalg.norm(J.T @ J, 2)
    ratios.append(ratio)
    costs.append(r_lm.value)
    gn_iters.append(r_gn.iterations if r_gn.converged else np.nan)
    lm_iters.append(r_lm.iterations if r_lm.converged else np.nan)
    print(f"{noise:7.2f} {r_lm.value:12.4e} {ratio:15.4e} "
          f"{r_gn.iterations:9d} {r_lm.iterations:9d}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].loglog(np.maximum(costs, 1e-26), ratios, "o-")
for noise, c, rr in zip(noises, costs, ratios):
    ax[0].annotate(f"$\\sigma$={noise}", (max(c, 1e-26), rr), fontsize=7,
                   textcoords="offset points", xytext=(5, -9))
ax[0].set_xlabel(r"$f(x^*) = \frac{1}{2}\|r\|^2$ at the solution")
ax[0].set_ylabel(r"$\|\sum r_i \nabla^2 r_i\| \,/\, \|J^\top J\|$")
ax[0].set_title("The dropped term grows with the residual")

ax[1].plot(noises, gn_iters, "o-", label="Gauss-Newton")
ax[1].plot(noises, lm_iters, "s--", label="Levenberg-Marquardt")
ax[1].set_xlabel(r"noise $\sigma$ in the data")
ax[1].set_ylabel("iterations to $\\|J^\\top r\\| \\leq 10^{-6}$")
ax[1].set_title("and the iteration count grows with it")
ax[1].legend(fontsize=8)

fig.suptitle("Figure 1 — 'valid for small residuals', quantified (easy start)")
fig.tight_layout()
plt.show()

**Figure 1.** With noiseless data the ratio is $2.3\times10^{-14}$ — the dropped term is
zero to rounding, because at a perfect fit every $r_i$ is zero and the sum has nothing
left in it. Gauss–Newton is then *exactly* Newton, and it converges in 4 iterations.

As the noise grows the ratio climbs monotonically to $2.5\times10^{-2}$ at $\sigma = 2$,
and the iteration count climbs with it, $4 \to 10$. Gauss–Newton has not broken; it has
degraded, smoothly and predictably, from quadratic towards linear convergence. The
textbook phrase "small residual" means this ratio, and the way to find out whether your
problem qualifies is to compute it.

Note also that GN and LM take **identical** iteration counts on every row. From a good
starting point LM's damping costs nothing at all — its $\lambda$ collapses immediately and
it *is* Gauss–Newton. Everything LM gains is in §3, at the starting points GN cannot
survive.

## 2. Gauss–Newton works, until it does not

Fix the noise at $\sigma = 0.05$ and vary the starting point instead. Four starts, all of
them things a person might reasonably type.

In [ ]:
problem, y_data = make_problem(0.05)

starts = {
    "easy  (4.0, 1.0, 0.5)": np.array([4.0, 1.0, 0.5]),
    "flat  (1.0, 0.1, 0.0)": np.array([1.0, 0.1, 0.0]),
    "steep (8.0, 5.0, 3.0)": np.array([8.0, 5.0, 3.0]),
    "hard  (0.5, 0.05, 4.0)": np.array([0.5, 0.05, 4.0]),
}

gn_runs, lm_runs = {}, {}
print(f"{'start':>24} {'method':>6} {'conv':>6} {'iters':>6} {'f':>12}   message")
for label, s in starts.items():
    h = History()
    r = GaussNewton(CholeskySolver(), max_iter=100, observers=[h]).minimize(problem, s)
    gn_runs[label] = (r, h)
    print(f"{label:>24} {'GN':>6} {str(r.converged):>6} {r.iterations:6d} "
          f"{r.value:12.4e}   {r.message}")

print()
for label, s in starts.items():
    h = History()
    r = LevenbergMarquardt(CholeskySolver(), max_iter=200, observers=[h]).minimize(problem, s)
    lm_runs[label] = (r, h)
    print(f"{label:>24} {'LM':>6} {str(r.converged):>6} {r.iterations:6d} "
          f"{r.value:12.4e}   {r.message}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))

ax[0].plot(t_grid, y_data, "k.", ms=4, label="data")
fine = np.linspace(0, 4, 300)
model = lambda tt, x: x[0] * np.exp(-x[1] * tt) + x[2]
for label, s in starts.items():
    ax[0].plot(fine, model(fine, s), "--", lw=1, label=f"start: {label.split()[0]}")
best = lm_runs["easy  (4.0, 1.0, 0.5)"][0].x
ax[0].plot(fine, model(fine, best), "k-", lw=2, label="fitted")
ax[0].set_xlabel("$t$")
ax[0].set_ylabel("$y$")
ax[0].set_title("The four starting guesses, as curves")
ax[0].legend(fontsize=7)

for label in starts:
    r_gn, h_gn = gn_runs[label]
    r_lm, h_lm = lm_runs[label]
    line, = ax[1].semilogy(np.arange(1, len(h_gn.values) + 1),
                           np.maximum(h_gn.values, 1e-3), "--", lw=1.2,
                           label=f"GN {label.split()[0]}")
    ax[1].semilogy(np.arange(1, len(h_lm.values) + 1),
                   np.maximum(h_lm.values, 1e-3), "-", lw=1.2,
                   color=line.get_color(), label=f"LM {label.split()[0]}")
    if not r_gn.converged:
        ax[1].plot(len(h_gn.values), max(h_gn.values[-1], 1e-3), "x",
                   color=line.get_color(), ms=11, mew=2)
ax[1].set_xlabel("iteration")
ax[1].set_ylabel(r"$f(x) = \frac{1}{2}\|r\|^2$")
ax[1].set_title("dashed: Gauss-Newton (x = failed),  solid: Levenberg-Marquardt")
ax[1].legend(fontsize=6.5, ncol=2)

fig.suptitle(r"Figure 2 — same data ($\sigma = 0.05$), four starting points")
fig.tight_layout()
plt.show()

**Figure 2.** Two of the four starts converge under Gauss–Newton (5 and 7 iterations) and
two fail — and they fail in *different* ways, which the `message` column spells out.

From `flat (1.0, 0.1, 0.0)` the message is **`normal equations failed`**. With $b$ near
zero the exponential is nearly constant over the whole window, so the columns of $J$ for
$a$ and $c$ become nearly parallel: $J^\top J$ loses rank, and your day-4
`CholeskySolver` refuses it. Read the reported $x$: $b$ has been thrown to
$1.8\times10^{5}$ by an earlier near-singular solve.

From `hard (0.5, 0.05, 4.0)` the message is **`iterates left the finite range`** — the
parameters reached `nan` in four iterations. Nothing was singular; the step was simply
enormous, because Gauss–Newton trusts its quadratic model over an unlimited distance and
that model is worthless this far from the solution.

Both failures come from the same root cause: the step $\delta = -(J^\top J)^{-1}J^\top r$
has no length control whatsoever. Notebook 2 solved that problem for gradient descent with
a line search. LM solves it a different way, and §4 shows why the difference matters.

Now look at the solid lines: **LM converges from all four**, to the same fit, in 5, 9, 7
and 10 iterations.

## 3. What $\lambda$ and $\rho$ are doing

LM solves $(J^\top J + \lambda I)\,\delta = -J^\top r$ and then asks whether the step was
any good, by comparing what actually happened to what the model promised:

$$\rho = \frac{f(x) - f(x + \delta)}{\text{predicted reduction}},
\qquad \text{predicted} = \tfrac{1}{2}\delta^\top(\lambda\delta - J^\top r) .$$

$\rho \approx 1$ means the model told the truth — accept, and trust it further by
*lowering* $\lambda$. $\rho \le 0$ means the step made things worse — reject it outright
and *raise* $\lambda$, which both shortens the next step and rotates it towards $-\nabla f$.

To see this you need the internals, and how you stored them in your own
`LevenbergMarquardt` is your business. So the loop below is written out here in full — it
is the same algorithm, using only `residuals`, `jacobian` and your `CholeskySolver`, and
it records every quantity as it goes. Compare it against your class.

In [ ]:
def lm_trace(prob, x0, lam=1e-3, lam_up=10.0, lam_down=0.1, tol=1e-8, max_iter=60):
    """Levenberg-Marquardt, written out so every decision is visible."""
    solver = CholeskySolver()
    x = np.asarray(x0, dtype=float).copy()
    r = prob.residuals(x)
    f = 0.5 * r @ r
    rows = []
    for it in range(1, max_iter + 1):
        J = prob.jacobian(x)
        grad = J.T @ r
        if np.linalg.norm(grad) <= tol:
            break
        delta = solver.solve(J.T @ J + lam * np.eye(x.size), -grad)
        x_new = prob.residuals(x + delta)
        f_new = 0.5 * x_new @ x_new
        predicted = 0.5 * delta @ (lam * delta - grad)
        rho = (f - f_new) / predicted if predicted > 0 else -1.0
        accepted = rho > 0.0
        rows.append({"it": it, "lam": lam, "rho": rho, "accepted": accepted,
                     "f": f, "grad": float(np.linalg.norm(grad)),
                     "step": float(np.linalg.norm(delta))})
        if accepted:
            x = x + delta
            r = prob.residuals(x)
            f = f_new
            lam = max(lam * lam_down, 1e-12)
        else:
            lam = lam * lam_up
    return x, rows


x_hard, rows_hard = lm_trace(problem, starts["hard  (0.5, 0.05, 4.0)"])
x_easy, rows_easy = lm_trace(problem, starts["easy  (4.0, 1.0, 0.5)"])

print("from the HARD start:")
print(f"  {'it':>3} {'lambda':>10} {'rho':>16} {'verdict':>9} {'f':>12} {'|step|':>11}")
for row in rows_hard:
    print(f"  {row['it']:3d} {row['lam']:10.2e} {row['rho']:16.4g} "
          f"{'accept' if row['accepted'] else 'REJECT':>9} {row['f']:12.4e} {row['step']:11.3e}")
print(f"\n  converged to {np.round(x_hard, 4)}   (true {x_true})")
print(f"  accepted {sum(r['accepted'] for r in rows_hard)} of {len(rows_hard)} attempts")
print(f"\nfrom the EASY start: {len(rows_easy)} attempts, "
      f"{sum(r['accepted'] for r in rows_easy)} accepted, "
      f"rho = {[round(r['rho'], 3) for r in rows_easy]}")

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(9, 8), sharex=True)

its = [r["it"] for r in rows_hard]
acc = np.array([r["accepted"] for r in rows_hard])

ax[0].semilogy(its, [r["f"] for r in rows_hard], "o-", color="0.3")
ax[0].set_ylabel(r"$f$ at the start of the step")
ax[0].set_title("Cost: flat across every rejected step, because $x$ does not move")

ax[1].semilogy(its, [r["lam"] for r in rows_hard], "o-", color="tab:blue")
ax[1].set_ylabel(r"$\lambda$")
ax[1].set_title(r"$\lambda$: up on every rejection, down on every acceptance")

rhos = np.array([r["rho"] for r in rows_hard])
shown = np.clip(rhos, -2.0, 2.0)
ax[2].plot(np.array(its)[acc], shown[acc], "o", color="tab:green", label="accepted")
ax[2].plot(np.array(its)[~acc], shown[~acc], "v", color="tab:red", label="rejected")
ax[2].axhline(1.0, color="k", ls="--", lw=1, label=r"$\rho = 1$: model was exact")
ax[2].axhline(0.0, color="k", lw=0.8)
ax[2].set_ylim(-2.3, 2.3)
ax[2].set_ylabel(r"$\rho$  (clipped to $[-2, 2]$)")
ax[2].set_xlabel("attempt")
ax[2].set_title(r"$\rho$: the accept/reject test")
ax[2].legend(fontsize=8)

for r in rows_hard:                    # label the points the clip hid
    if r["rho"] < -2.0:
        ax[2].annotate(f"$\\rho$ = {r['rho']:.1e}", (r["it"], -2.0), fontsize=7,
                       color="tab:red", textcoords="offset points", xytext=(4, 6))
    elif r["rho"] > 2.0:
        ax[2].annotate(f"$\\rho$ = {r['rho']:.3g}", (r["it"], 2.0), fontsize=7,
                       color="tab:green", textcoords="offset points", xytext=(-38, -12))

fig.suptitle("Figure 3 — every decision LM made from the hard start")
fig.tight_layout()
plt.show()

**Figure 3, and the table above it.** Read attempt 4. Its $\rho$ is
$-5\times10^{19}$ — the step the model promised would help made the cost worse by twenty
orders of magnitude. **That is the exact step that sent Gauss–Newton to `nan` in §2.** LM
computed the same catastrophic $\delta$, measured what it actually did, threw it away, and
raised $\lambda$. That is the entire difference between the two methods.

The rest of the trace is the mechanism in miniature. Attempts 1 and 2 are rejected with
$\rho = -85$ and $-1.5$, and $\lambda$ climbs $10^{-3} \to 10^{-1}$, shrinking the step
from $34.5$ to $2.3$. Attempt 3 is accepted at $\rho = 0.34$ — mediocre, but genuine
progress. From attempt 5 on, $\rho$ sits at $0.99, 1.00, 1.005, 1.008, 1.009$: the
quadratic model is now essentially exact, $\lambda$ falls by a factor of ten each time,
and LM has turned back into Gauss–Newton for the fast endgame. Seven of ten attempts were
accepted; the three rejections cost one extra residual evaluation each and bought the
whole run.

The final attempt has $\rho = 8.12$, far above 1. That is not a problem — it means the
step did *eight times better* than the model promised, which at a step length of
$3.6\times10^{-9}$ is a ratio of two quantities that are both essentially zero. This is
why the accept test is $\rho > 0$ and not $\rho \approx 1$: near the solution $\rho$
becomes numerical noise, and §4 is what happens when you lean on it too hard.

The middle panel is worth staring at. $\lambda$ is not a tuning parameter here; it is a
*measurement device* that reads out how far the quadratic model can currently be trusted.
That is why the easy start needs no help at all — every one of its five $\rho$ values is
within 1.6% of 1, so $\lambda$ only ever falls.

> **Line search versus trust region.** A line search fixes the direction and searches
> along it. LM changes $\lambda$, which changes the direction *and* the length together:
> as $\lambda$ grows, $\delta \to -\nabla f/\lambda$. It is searching over a family of
> directions, which is exactly what you need when the Gauss–Newton direction itself is
> the problem.

## 4. Breaking Levenberg–Marquardt

LM is robust, not magic. Here is a failure you are likely to meet this afternoon.

Ask for a tolerance tighter than the arithmetic can deliver. Once $x$ is correct to the
last bit, the *actual* reduction $f(x) - f(x + \delta)$ is exactly $0$ in float64 — the
cost cannot go down any further because it is already at its representable minimum. So
$\rho = 0$, which is not $> 0$, so the step is rejected. $\lambda$ is multiplied by
`lambda_up`, which makes the next step shorter, which makes the reduction even more
exactly zero. And so on.

Note the shape of the bug: the accept test is a *strict* inequality on a quantity that
converges to zero. Nothing is wrong with any individual line.

In [ ]:
x_trap, rows_trap = lm_trace(problem, easy_start, tol=1e-14, max_iter=60)

print(f"requested tol 1e-14; ran {len(rows_trap)} attempts, "
      f"{sum(r['accepted'] for r in rows_trap)} accepted")
print(f"final x  {np.round(x_trap, 6)}")
print(f"final |grad| {rows_trap[-1]['grad']:.3e}   final lambda {rows_trap[-1]['lam']:.3e}")
print()
print(f"  {'it':>3} {'lambda':>11} {'rho':>10} {'verdict':>8} {'|grad|':>11} {'|step|':>11}")
for row in rows_trap[:4] + rows_trap[-5:]:
    print(f"  {row['it']:3d} {row['lam']:11.2e} {row['rho']:10.3g} "
          f"{'accept' if row['accepted'] else 'REJECT':>8} {row['grad']:11.3e} {row['step']:11.3e}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

its_t = [r["it"] for r in rows_trap]
ax[0].semilogy(its_t, [r["lam"] for r in rows_trap], "o-", ms=3)
ax[0].set_xlabel("attempt")
ax[0].set_ylabel(r"$\lambda$")
ax[0].set_title(r"$\lambda$ runs away once every step is rejected")

ax[1].semilogy(its_t, np.maximum([r["grad"] for r in rows_trap], 1e-20), "o-", ms=3,
               label=r"$\|J^\top r\|$")
ax[1].axhline(1e-14, color="tab:red", ls="--", lw=1, label="requested tol $10^{-14}$")
ax[1].axhline(rows_trap[-1]["grad"], color="k", ls=":", lw=1,
              label=f"best reached: {rows_trap[-1]['grad']:.1e}")
ax[1].set_xlabel("attempt")
ax[1].set_ylabel(r"$\|J^\top r\|$")
ax[1].set_title("the gradient stopped improving long before")
ax[1].legend(fontsize=8)

fig.suptitle("Figure 4 — the same easy start, with an impossible tolerance")
fig.tight_layout()
plt.show()

**Figure 4.** The answer is correct — `final x` matches the §2 fit to six decimals — but
the run burns all 60 attempts and reports failure. Only the first 7 did anything; the
other 53 were rejections of a step that was already right.

Read the two tables together. By attempt 7 the gradient has reached $5.75\times10^{-13}$
and it never improves again, while $\lambda$ climbs by a factor of ten every attempt:
$10^{-3}$ at the start, $10^{42}$ at attempt 60. The step length falls in lockstep,
reaching $5.8\times10^{-55}$ — LM is now proposing moves 40 orders of magnitude below
anything that could change a float64. Give it 60 more attempts and $\lambda$ reaches
`inf`, $(J^\top J + \lambda I)$ becomes `inf` on the diagonal, $\delta$ becomes zero, and
the loop is stuck permanently. $\lambda \to \infty$ is an **absorbing state**: there is no
value of $\rho$ that can ever bring it back down.

Three fixes, all one line, in increasing order of how much I would trust them:

1. **Stop on the step too**, not only on the gradient: if $\|\delta\|$ is below, say,
   $10^{-12}(1 + \|x\|)$, you have converged, whatever $\rho$ says. This is what most
   production LM codes do, and it is the right fix — it addresses the actual condition.
2. **Cap $\lambda$.** If $\lambda$ exceeds something like $10^{12}$, stop and report that
   no acceptable step exists. Cheap, and it converts an infinite loop into a message.
3. **Ask for a reachable tolerance.** $\|J^\top r\| \le 10^{-14}$ was never going to
   happen at this scale. Worth knowing, but it treats the symptom.

Your `LevenbergMarquardt` may already do one of these, in which case this cell will print
something tidier — read your own output rather than this paragraph. Either way the lesson
holds: **a rejection rule that can reject forever needs a second way out.**

## 5. The same two classes on two other models

Nothing in `GaussNewton` or `LevenbergMarquardt` mentions exponentials. They take a
`ILeastSquaresProblem`, and `GaussianPeak` and `Sinusoid` are two more of those. If the
abstraction is right, fitting them is a change of *data*, not of code.

`Sinusoid` is the interesting one. Its docstring warns that $\omega$ has many local
minima — get the frequency wrong by a whole cycle and the residuals look just as good
locally. Watch what that does, and read the `f` column rather than the `conv` column.

In [ ]:
peak_t = np.linspace(-3.0, 5.0, 80)
peak_rng = np.random.default_rng(17)
peak_y = 3.0 * np.exp(-(peak_t - 1.2) ** 2 / (2 * 0.8 ** 2)) + peak_rng.normal(scale=0.05, size=peak_t.size)
peak = GaussianPeak(peak_t, peak_y)

sin_t = np.linspace(0.0, 6.0, 120)
sin_rng = np.random.default_rng(23)
sin_y = 2.0 * np.sin(3.0 * sin_t + 0.7) + sin_rng.normal(scale=0.1, size=sin_t.size)
sinus = Sinusoid(sin_t, sin_y)

print("GaussianPeak, true (a, mu, sigma) = (3.0, 1.2, 0.8)")
for s in ([1.0, 0.0, 1.0], [5.0, 4.0, 3.0]):
    r = LevenbergMarquardt(CholeskySolver(), max_iter=300).minimize(peak, np.array(s))
    print(f"  from {s}: conv {str(r.converged):5s} iters {r.iterations:3d}  -> {np.round(r.x, 4)}")

print("\nSinusoid, true (a, omega, phi) = (2.0, 3.0, 0.7)")
for w0 in [2.8, 3.0, 3.4, 5.0, 8.0]:
    s = np.array([1.5, w0, 0.0])
    r = LevenbergMarquardt(CholeskySolver(), max_iter=300).minimize(sinus, s)
    print(f"  omega start {w0:4.1f}: conv {str(r.converged):5s} iters {r.iterations:3d}  "
          f"-> a {r.x[0]:7.4f}  omega {r.x[1]:7.4f}  phi {r.x[2]:8.4f}   f {r.value:.4e}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(peak_t, peak_y, "k.", ms=4, label="data")
r_peak = LevenbergMarquardt(CholeskySolver(), max_iter=300).minimize(peak, np.array([1.0, 0.0, 1.0]))
fine_p = np.linspace(-3, 5, 300)
ax[0].plot(fine_p, r_peak.x[0] * np.exp(-(fine_p - r_peak.x[1]) ** 2 / (2 * r_peak.x[2] ** 2)),
           "-", lw=2, label=f"LM fit {np.round(r_peak.x, 3)}")
ax[0].set_xlabel("$t$")
ax[0].set_title("GaussianPeak: the same two classes, no edits")
ax[0].legend(fontsize=8)

ax[1].plot(sin_t, sin_y, "k.", ms=3, label="data")
fine_s = np.linspace(0, 6, 600)
for w0, style in ((3.0, "-"), (8.0, "--"), (5.0, ":")):
    r = LevenbergMarquardt(CholeskySolver(), max_iter=300).minimize(sinus, np.array([1.5, w0, 0.0]))
    ax[1].plot(fine_s, r.x[0] * np.sin(r.x[1] * fine_s + r.x[2]), style, lw=1.5,
               label=f"start $\\omega$={w0} $\\rightarrow$ {r.x[1]:.3f},  f={r.value:.3g}")
ax[1].set_xlabel("$t$")
ax[1].set_title(r"Sinusoid: one right answer, one duplicate, one disaster")
ax[1].legend(fontsize=7.5)

fig.suptitle("Figure 5 — the abstraction earns its keep, and finds its limit")
fig.tight_layout()
plt.show()

**Figure 5, left.** `GaussianPeak` fits from both starting points with no change to any
optimizer. This is the payoff for `ILeastSquaresProblem` being an interface rather than a
base class with an exponential baked into it.

**Figure 5, right.** Three outcomes from five starting frequencies, and each teaches
something different.

Starts at $\omega = 2.8, 3.0, 3.4$ all land on $\omega = 3.0026$ with $f = 0.7015$ —
`converged=True`, in 12, 6 and 6 iterations. That is the answer.

The start at $\omega = 8.0$ finishes with $\omega = -3.0026$ and $f = 0.7015$ — the *same
cost, to four decimals*. It is not a worse fit; look at the plot, the dashed curve lies
exactly on the solid one. It is the **same curve** reached through a different parameter
vector, because $a\sin(\omega t + \varphi)$ is unchanged by flipping the sign of $\omega$
and adjusting $\varphi$. The model is not *identifiable*: distinct parameters, identical
predictions, so the minimum is not unique and no optimizer can prefer one over the other.
That is a property of how you parameterised the model, and the fix is in the model —
constrain $\omega > 0$ — not in the optimizer.

The start at $\omega = 5.0$ is the genuine disaster: $f = 124.7$ against $0.70$, with the
amplitude collapsed to $a = 0.09$. LM slid into a flat region, could not find the right
frequency from there, and gave up on a nearly-zero curve. The dotted line in the plot is
essentially flat.

Both bad runs report `converged=False` after exhausting 300 iterations, which at least
makes them visible — but do not count on that. `converged=True` only ever means "the
gradient is small *here*". On a non-convex problem the starting point selects the answer,
and the only defence is the one the optimizer will never suggest: a coarse grid of
starting values, LM from each, and keep the lowest final cost.

## Checkpoint

1. A colleague's Gauss–Newton reports `normal equations failed` on their own data. Using
   §2, name two structurally different causes, and say which measurement distinguishes
   them.
2. In §3 LM computed the same disastrous step as Gauss–Newton. Which single number saved
   it, and what would have happened had that number been computed as
   `(f - f_new) / f` instead?
3. Your LM converges in 5 iterations from every start you try, and $\lambda$ never rises.
   Is the damping doing anything? What does Figure 1 say you should expect from
   Gauss–Newton on that problem?
4. Figure 1's ratio is $2\times10^{-2}$ on your data. Would you use Gauss–Newton, LM, or
   full Newton with the exact Hessian, and what would you measure to decide?
5. Write the one-line stopping test from §4 fix 1 into your own `LevenbergMarquardt`,
   rerun the impossible-tolerance cell, and check that it now reports success with the
   same `x`.

**Before day 6.** Notebook 6 needs `L1.prox` (the soft-threshold), `ProximalGradient`,
and the `Huber` and `PoissonNLL` pointwise losses. Note that `L1.gradient` must keep
raising `NotImplementedError` — that is the contract, not a gap, and notebook 6 §1 makes
the reason visible.